# **MLLM-SHAP: Practice**

Hi, friends! In the lesson "How do we remove features in different modalities? Mask effects" we went through SGPA: audio is cut into words, a switched-off word is replaced with silence, and Shapley values are estimated with complementary contributions $CC(S)=v(S)-v(N\setminus S)$ and Neyman allocation of samples across coalition sizes. The lesson "SHAP for generative models" added a second idea: when the model output is not a number, $v(S)$ has to be defined by comparing answers, and choosing the value function means choosing the game itself.

Today we test both ideas with the [`mllm-shap`](https://github.com/Pawlo77/MLLM-SHAP) library (Paweł Pozorski, [documentation](https://pawlo77.github.io/MLLM-SHAP/)), which implements SGPA and the estimators from the lesson. We will explain a small open chat model, `Qwen2.5-0.5B-Instruct`: it runs on a Colab CPU and needs neither an API key nor a Hugging Face token.

In this practice you will:
- compute exact Shapley values for the words of a prompt with the `exact` estimator and check efficiency: $\sum_i\phi_i = v(N)-v(\varnothing)$;
- collect the table of $v(S)$ for all 128 coalitions and recompute the Shapley values from it yourself;
- compare three approximate estimators with the exact answer — Monte Carlo, complementary, and complementary with Neyman allocation — at different budgets of model runs;
- see how the attribution changes when the value function changes, and define your own: the log-probability of the answer "Paris";
- cut a speech recording into words with SGPA and switch some words off with silence;
- build PixelSHAP by hand, with the objects of an image as players for a small VLM, and check how accurate the first-order approximation is.

Let's start!

First, the dependencies. Only `mllm-shap` is missing on Colab; we pin its version so that the numbers match ours. `torch`, `transformers`, `librosa` and `soundfile` are already installed on Colab. There is no need for the `mllm-shap[viz]` extra: it requires `ipywidgets>=8.1` and would upgrade the Colab version, while the library's plots only need plain `matplotlib`.

In [ ]:
!pip install mllm-shap==0.3.0 -q

In [ ]:
import random
import urllib.request
from itertools import combinations
from math import comb, factorial

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer

import mllm_shap
from mllm_shap import (AudioPart, CoalitionMask, ConversationSnapshot, DeletePolicy, ExplainConfig,
                       ExplainRunner, Message, NeutralPolicy, PluginContext, Role, SgpaSegmentPartitioner,
                       TextPart, TransformersTextBackend, Turn)
from mllm_shap.alignment import SpectrogramGuidedAligner, Wav2Vec2Ctc, decode_audio
from mllm_shap.viz import plot_audio_segments


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('mllm-shap', mllm_shap.__version__, '| device:', device)

## The game: prompt words and the model's answer

In `mllm-shap` an explanation is assembled from interchangeable parts, and each part is one decision from the lessons:

| config part | what it sets | what we use today |
|---|---|---|
| `players` | who the players are | `tokens` — words of the user message (split on whitespace) |
| `absence_policy` | how a player is "switched off" | `delete` — the word is removed from the string |
| `value_fn` | the value function $v(S)$ | `logprob`, then `tfidf_cosine` and `embedding_cosine` |
| `estimator` | how coalitions are enumerated | `exact`, `mc`, `complementary`, `neyman_cc` |
| `backend` | whose model answers | `transformers` — a local Hugging Face model |

The conversation is a system message "Answer with one word." and the user's question "What is the capital city of France?". The players are the 7 words of the question; the system message is present in every coalition. Generation is greedy (`temperature=0`, `top_k=1`) and at most 6 tokens long, so the same truncated prompt always gives the same answer. We pin the model revision on Hugging Face to a commit hash.

Two technical details. The `transformers` backend picks the device itself (CUDA if available, otherwise CPU); the `device` variable above is only for our own code in Task 3. And in Jupyter `ExplainRunner` is called as `await runner.explain(...)`: the synchronous `explain_sync` is not allowed inside Jupyter's already running event loop.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
SYSTEM = 'Answer with one word.'
PROMPT = 'What is the capital city of France?'

snapshot = ConversationSnapshot.from_turns((Turn(messages=(
    Message(role=Role.SYSTEM, text=SYSTEM),
    Message(role=Role.USER, text=PROMPT),
)),))


def make_config(estimator='exact', value_fn='logprob', evaluations=None, seed=0):
    # exact enumerates all 2^n coalitions; approximate estimators get a number of runs, evaluations
    budget = {'fraction': 1.0} if evaluations is None else {'evaluations': evaluations}
    return ExplainConfig.model_validate({
        'backend': {'kind': 'transformers', 'model_id': MODEL_ID,
                    'options': {'revision': REVISION, 'dtype': 'float32'}},
        'generation': {'max_new_tokens': 6, 'temperature': 0.0, 'top_k': 1},
        'explanation': {'estimator': estimator, 'value_fn': value_fn, 'budget': budget, 'seed': seed,
                        'archive': {'path': ''},                            # do not write a run archive to disk
                        'plugin_options': {'tokens': {'roles': ['user']}}},  # players are the question words only
    })

Below we explain the same prompt many times: with different estimators, budgets and seeds, and different value functions. Each run would generate answers to the same truncated prompts again. So we wrap the backend in a cache: the answer to every truncated prompt is generated once, and repeated requests are served from the dictionary `backend.records`. Generation is greedy, so a cached answer equals what the model would return on a repeated call. The wrapper also always asks the backend for answer embeddings: the `embedding_cosine` value function needs them, and they do not affect the others.

`ExplainRunner` accepts a ready backend through the `backend=` argument; for the library it is just an object with a `model_id` field and an asynchronous `generate` method.

In [ ]:
class CachedBackend:
    def __init__(self, inner):
        self.inner = inner
        self.records = {}      # (role, text) of every message of a truncated prompt -> model answer
        self.model_calls = 0   # how many times the model actually generated

    @property
    def model_id(self):
        return self.inner.model_id

    @property
    def returns_logprobs(self):
        return True

    async def generate(self, snap, generation):
        key = tuple((m.role.value, m.text) for m in snap.messages())
        if key not in self.records:
            self.model_calls += 1
            self.records[key] = await self.inner.generate(
                snap, generation.model_copy(update={'compute_embeddings': True}))
        return self.records[key]


backend = CachedBackend(TransformersTextBackend(PluginContext(config=make_config())))


async def explain(**kwargs):
    return await ExplainRunner(make_config(**kwargs), backend=backend).explain(snapshot)

## The exact estimator and efficiency

The first value function is `logprob`. For a coalition $S$ the model answers the prompt that keeps only the words in $S$, and $v(S)$ is the mean log-probability of the tokens of that answer:

$$v(S) = \frac{1}{|g(x_S)|}\sum_{t} \log p\big(g_t(x_S)\mid x_S, g_{<t}(x_S)\big),$$

where $g(x_S)$ is the model's answer to the truncated prompt $x_S$. Note that this is the model's confidence in **its own** answer to the truncated prompt, not the probability of the answer "Paris". Every coalition has its own answer, so $v(S)$ says how confidently the model answers anything at all. We will define the probability of a specific answer ourselves in Task 3.

The `exact` estimator enumerates all $2^7=128$ coalitions — on a Colab CPU that takes about a minute or two. Shapley values are then computed with the classical formula.

In [ ]:
exact = await explain(estimator='exact', value_fn='logprob')
res = exact.result
words = [span.text for span in exact.player_set.spans]
n = len(words)
phi_exact = np.array(res.values)

print('players:', words)
print('model calls:', backend.model_calls, '| answer to the full prompt:', repr(backend.records[(('system', SYSTEM), ('user', PROMPT))].text))
print(f'v(empty) = {res.base_value:.4f}, v(N) = {res.grand_value:.4f}')
pd.DataFrame({'phi (logprob)': phi_exact}, index=words).round(4)

The result stores not only the contributions `values` but also both ends of the game: `base_value` $=v(\varnothing)$ and `grand_value` $=v(N)$. This lets us check the efficiency axiom in one line.

In [ ]:
print(f'sum of phi     = {phi_exact.sum():.6f}')
print(f'v(N) - v(empty) = {res.grand_value - res.base_value:.6f}')
print('efficiency holds:', np.isclose(phi_exact.sum(), res.grand_value - res.base_value))

In [ ]:
def plot_words(values, title, xlabel):
    colors = ['tab:red' if v > 0 else 'tab:blue' for v in values]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(words, values, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel(xlabel)
    ax.set_title(title)
    plt.show()


plot_words(phi_exact, 'Exact Shapley values of the prompt words, value_fn = logprob',
           'phi: contribution to the mean\nlog-probability of the answer')

Red bars raise the model's confidence in its answer, blue bars lower it. Let us also look at the answers themselves: what the model says when only one word of the question is left.

In [ ]:
def masked_snapshot(present):
    # the same truncated prompt the library gives the model for the coalition `present`
    return DeletePolicy().apply(snapshot, exact.player_set, CoalitionMask(tuple(present)))


def record_of(present):
    return backend.records[tuple((m.role.value, m.text) for m in masked_snapshot(present).messages())]


for i in range(n):
    present = [j == i for j in range(n)]
    print(f'{{{words[i]}}}'.ljust(12), '->', repr(record_of(present).text))
print('{}'.ljust(12), '->', repr(record_of([False] * n).text))

**Self-check question.** Which word got the largest Shapley value in the `logprob` game, and what is it? What are the sum of the Shapley values and $v(N)-v(\varnothing)$?

<details><summary>Check yourself</summary>

The largest Shapley value goes to the word `France?`: $\phi = 0.4479$, followed by `city` with $0.2985$. The sum of the Shapley values is $1.0306$, and $v(N)-v(\varnothing) = -0.0530-(-1.0836) = 1.0306$: efficiency holds. The words `the` ($-0.0263$) and `of` ($-0.0144$) get negative contributions.

</details>

## Task 1. Shapley values from the table $v(S)$

All 128 answers are already in the cache `backend.records`, and the function `record_of(present)` returns the answer for any coalition. Below they are collected into the table `v_logprob`: the key is the set of word indices in the coalition (`frozenset`), the value is the mean log-probability of the answer, exactly as `value_fn='logprob'` computes it (an empty answer gives 0).

Write a function `shapley_exact(v, n)` that computes the Shapley values of all $n$ players from such a table with the formula

$$\phi_i = \sum_{S\subseteq N\setminus i}\frac{|S|!\,(n-|S|-1)!}{n!}\,\big(v(S\cup i)-v(S)\big),$$

and compare the result with `phi_exact` from the library. You will need the function again in Task 3.

In [ ]:
def mean_logprob(record):
    return sum(record.logprobs) / len(record.logprobs) if record.logprobs else 0.0


v_logprob = {frozenset(S): mean_logprob(record_of([i in S for i in range(n)]))
             for size in range(n + 1) for S in combinations(range(n), size)}
print('coalitions in the table:', len(v_logprob))

In [ ]:
def shapley_exact(v, n):
    # Your code here: for every player i enumerate the coalitions S without i (combinations),
    # add the weight |S|!(n-|S|-1)!/n! times v[S | {i}] - v[S]
    ...


phi_manual = shapley_exact(v_logprob, n)
print(pd.DataFrame({'by hand': phi_manual, 'mllm-shap exact': phi_exact}, index=words).round(4))
print('max |difference|:', np.abs(phi_manual - phi_exact).max())

**Self-check question.** What is the order of magnitude of the maximum absolute difference between your values and the `exact` values? What does it tell you?

<details><summary>Check yourself</summary>

The maximum absolute difference is about $10^{-16}$ ($1.1\cdot10^{-16}$ in our run), i.e. at the level of floating-point rounding. The `exact` estimator is the same classical Shapley formula applied to the same table of 128 values $v(S)$; only the order of summation differs.

</details>

## Task 2. Approximate estimators at different budgets

`mllm-shap` has three approximate estimators. The budget `evaluations` is the total number of model runs, including the empty and the full coalitions.

- **`mc`** — Monte Carlo: random coalitions; a word's contribution is the difference between Shapley-weighted means of $v$ "with the word" and "without the word".
- **`complementary`** — complementary contributions (Zhang et al., 2023): coalitions come in pairs $S$ and $N\setminus S$, and one pair $CC(S)=v(S)-v(N\setminus S)$ updates the estimates of all $n$ players at once. The minimum budget is $2n+2=16$: the pair $(N,\varnothing)$ and the pairs $(\{i\}, N\setminus\{i\})$ for every word.
- **`neyman_cc`** — the same estimator with the two-phase scheme from SGPA: in phase 1 a small equal sample per stratum (coalition size) estimates the spread $\sigma_{i,j}$, in phase 2 the rest of the budget is split across strata in proportion to $\sigma$.

Let us compare them with the exact answer `phi_exact`. For every estimator, every budget in `budgets` and every seed in `seeds`, run `await explain(estimator=..., value_fn='logprob', evaluations=..., seed=...)` and compute two quantities: the mean absolute error $\frac1n\sum_i|\hat\phi_i-\phi_i|$ and the Pearson correlation between $\hat\phi$ and `phi_exact`. Average both over seeds and put them into the table `errors` with columns `estimator`, `budget`, `MAE`, `pearson`.

The value function here is `logprob` on purpose: $v(S)$ depends only on the answer to the coalition $S$. This is not true for `tfidf_cosine` — its IDF weights are fitted on all answers of one run, so an approximate estimator that saw only part of the coalitions would get a slightly different game.

The model generates almost no new answers: all 128 coalitions are already cached, so the cell takes seconds. The budget is capped by the number of coalitions, $2^7=128$.

In [ ]:
estimators = ['mc', 'complementary', 'neyman_cc']
budgets = [20, 32, 48, 64, 96, 128]
seeds = range(10)

rows = []
# Your code here: a triple loop over estimators, budgets and seeds; for every pair (estimator, budget)
# append to rows a dict {'estimator', 'budget', 'MAE', 'pearson'} with the means over seeds
errors = pd.DataFrame(rows)
print('model calls in total:', backend.model_calls)
errors.pivot(index='budget', columns='estimator', values=['MAE', 'pearson']).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for estimator in estimators:
    part = errors[errors.estimator == estimator]
    axes[0].plot(part.budget, part.MAE, marker='o', label=estimator)
    axes[1].plot(part.budget, part.pearson, marker='o', label=estimator)
axes[0].set_ylabel('mean absolute error of phi ↓')
axes[1].set_ylabel('Pearson correlation with exact ↑')
for ax in axes:
    ax.set_xlabel('budget: number of model runs')
    ax.axvline(2 ** n, color='gray', linestyle='--', linewidth=0.8)
    ax.legend()
fig.suptitle(f'Approximate estimators against exact, value_fn = logprob, mean over {len(seeds)} seeds; '
             f'dashed line: 2^n = {2 ** n} coalitions')
plt.tight_layout()
plt.show()
print(f'SGPA budget 3n^2 = {3 * n ** 2}, full enumeration 2^n = {2 ** n}')

**Self-check question.** Which estimator gives the smallest mean absolute error at a budget of 32 runs, and what is it? What happens to the complementary estimators at a budget of 128? And compare the SGPA budget $3n^2$ with full enumeration $2^n$ for our 7 words: when does an approximate estimator make sense at all?

<details><summary>Check yourself</summary>

At a budget of 32 runs the smallest mean absolute error belongs to `complementary`: $0.0738$ (`neyman_cc` $0.0807$, `mc` $0.1454$; mean over 10 seeds). Neyman allocation beats `complementary` at budgets of 20 and 64: $0.0827$ against $0.1003$ and $0.0401$ against $0.0420$. At a budget of 128 all three estimators have zero error and correlation 1: 128 = $2^7$, they see every coalition. The SGPA budget $3n^2 = 147$ for 7 words exceeds full enumeration $2^7 = 128$, so here the exact estimator is cheaper. An approximate estimator makes sense when $3n^2 < 2^n$, i.e. from 8 players on ($192 < 256$); with 10 words it is 300 runs against 1024.

</details>

## Choosing the value function means choosing the game

The lesson "SHAP for generative models" splits value functions into two strategies: **similarity** of the answer to a truncated prompt to the answer to the full prompt, and **probability**. `mllm-shap` has both:

- `tfidf_cosine` — the cosine between TF-IDF vectors of the coalition's answer and of the full prompt's answer, as in TokenSHAP; by construction $v(N)=1$;
- `embedding_cosine` — the cosine between the mean input embeddings of the tokens of the two answers, as in PixelSHAP;
- `logprob` — the mean log-probability of the model's own answer; we have already computed it.

We compute the Shapley values of the same 7 words in all three games with the exact estimator. All answers are cached, the model is not called again: only the way answers turn into the number $v(S)$ changes.

In [ ]:
games = {'logprob': res}
for value_fn in ['tfidf_cosine', 'embedding_cosine']:
    games[value_fn] = (await explain(estimator='exact', value_fn=value_fn)).result
print('model calls in total:', backend.model_calls)

table = pd.DataFrame({name: r.values for name, r in games.items()}, index=words)
bounds = pd.DataFrame({name: [r.base_value, r.grand_value, sum(r.values)] for name, r in games.items()},
                      index=['v(empty)', 'v(N)', 'sum of phi'])
print(bounds.round(4), '\n')
table.round(4)

The values of the three games are measured in different units, so for the plot we divide each column by its $v(N)-v(\varnothing)$: the bars then show the share of the game's total payoff each word received, summing to 1 in every game.

In [ ]:
share = table / bounds.loc['v(empty)'].rsub(bounds.loc['v(N)'])
pos = np.arange(n)
fig, ax = plt.subplots(figsize=(10, 4))
for k, name in enumerate(share.columns):
    ax.bar(pos + (k - 1) * 0.27, share[name], width=0.27, label=name)
ax.set_xticks(pos, words)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('phi / (v(N) - v(empty))')
ax.set_title('Share of the game payoff per word in three games (exact)')
ax.legend(title='value_fn')
plt.show()
print(share.round(3), '\n')

rho = pd.DataFrame([[spearmanr(table[a], table[b])[0] for b in table] for a in table], index=table.columns, columns=table.columns)
print('Spearman rank correlation of words between games:')
rho.round(3)

Efficiency holds in all three games: the sum of the Shapley values equals $v(N)-v(\varnothing)$. But the meaning of that sum differs. In `tfidf_cosine` and `embedding_cosine` what is shared is similarity to the answer "Paris": a word gets more the more it keeps the answer similar to the full one. In `logprob` what is shared is the model's confidence in whatever it answers. These are three different games on the same 128 answers.

**Self-check question.** Which word gets the largest share of the payoff in the `tfidf_cosine` game, and which in the `logprob` game? Which pair of games disagrees most on the ranking of words (smallest Spearman correlation)?

<details><summary>Check yourself</summary>

In the `tfidf_cosine` game the largest share goes to `France?` ($0.644$), in the `logprob` game also to `France?` ($0.435$), and in the `embedding_cosine` game to `capital` ($0.273$; `France?` gets $0.223$ there). `logprob` and `embedding_cosine` disagree most: the Spearman correlation of word ranks is $0.179$. The two similarity games, `tfidf_cosine` and `embedding_cosine`, agree most: $0.857$.

</details>

## Task 3. Your own value function: the probability of the answer "Paris"

None of the built-in value functions asks how probable a **specific** answer is. Let us define a probability game, as in TextGenSHAP from the lesson "SHAP for generative models":

$$v(S) = \log p\big(\text{"Paris"} \mid x_S\big) = \sum_t \log p\big(y_t \mid x_S, y_{<t}\big),$$

where $y$ are the tokens of the answer to the full prompt. The model generates nothing here: the answer is inserted as a whole, and one forward pass gives the probability of each of its tokens (teacher forcing).

Load the same model with `transformers` and write a function `target_logprob(user_text)`: build the prompt with the chat template (`tokenizer.apply_chat_template(..., add_generation_prompt=True)`) from the system message and the truncated question, append the tokens of the answer `TARGET`, run a forward pass and sum the log-probabilities of the answer tokens. The truncated question for a coalition is `masked_snapshot(present).messages()[1].text`. Then build the table `v_target` over all 128 coalitions, compute the Shapley values with your `shapley_exact` from Task 1 and check efficiency.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
lm = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32).to(device).eval()
TARGET = backend.records[(('system', SYSTEM), ('user', PROMPT))].text
print('target answer:', repr(TARGET))

In [ ]:
target_ids = tokenizer(TARGET, add_special_tokens=False, return_tensors='pt').input_ids.to(device)


def target_logprob(user_text):
    # Your code here: chat template -> prompt tokens, append target_ids, forward pass of lm under torch.no_grad(),
    # sum of log_softmax over the answer tokens (the logit at position t predicts token t + 1)
    ...


v_target = # Your code here: a dict frozenset(S) -> target_logprob(truncated question) over all coalitions
phi_target = shapley_exact(v_target, n)

v_empty, v_full = v_target[frozenset()], v_target[frozenset(range(n))]
print(f'log p(answer | empty question) = {v_empty:.4f}, p = {np.exp(v_empty):.4f}')
print(f'log p(answer | full question)  = {v_full:.4f}, p = {np.exp(v_full):.4f}')
print(f'p(answer | full) / p(answer | empty) = {np.exp(v_full - v_empty):.0f}')
print(f'sum of phi = {phi_target.sum():.4f}, v(N) - v(empty) = {v_full - v_empty:.4f}')
pd.DataFrame({'phi: log p(Paris)': phi_target}, index=words).round(4)

In [ ]:
plot_words(phi_target, 'Exact Shapley values of the prompt words, v(S) = log p(Paris | x_S)',
           'phi: contribution to\nlog p(Paris)')
print('Spearman with the library games:', {name: round(float(spearmanr(phi_target, table[name])[0]), 3) for name in table})

**Self-check question.** By what factor does the full question raise the probability of the answer "Paris" compared with the empty one? Which word contributes most to $\log p(\text{Paris})$, and which of the three library games does this game agree with best on the ranking of words?

<details><summary>Check yourself</summary>

The probability of the answer "Paris" grows from $0.0001$ ($\log p = -8.9051$) for the empty question to $0.9891$ ($\log p = -0.0110$) for the full one, about 7289 times. The largest contribution belongs to `France?`: $\phi = 5.7210$; the word `is` has a negative contribution, $-2.3204$. The sum of the Shapley values is $8.8942 = v(N)-v(\varnothing)$. On word ranks this game agrees best with `tfidf_cosine`: Spearman correlation $0.964$ (with `embedding_cosine` $0.821$, with `logprob` $0.571$).

</details>

## Audio: SGPA cuts speech into words

In version 0.3.0 the library has the audio part of SGPA that turns a recording into word players and the silence mask: the `sgpa_segments` partition (CTC alignment with `facebook/wav2vec2-base-960h` plus boundary refinement by energy and spectral flux with weights 0.8 and 0.2) and absence policies that cut or silence the samples of switched-off words. There is no model that would answer audio: the `transformers`, `lmstudio` and `api` backends refuse audio messages, and `mock` does not build an answer to sound (this is the open issue D02 in the library repository). So Shapley values for an audio LLM cannot be computed in this version, but we can look at the alignment and the masking.

The recording is the phrase "I had that curiosity beside me at this moment" from the VOiCES corpus (CC BY 4.0 license); a copy from the library repository (tag v0.3.0) is stored in the `data/` folder of the course repository. The alignment model is about 360 MB.

In [ ]:
WAV_URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/voices_sp0307.wav'
CTC_MODEL, CTC_REVISION = 'facebook/wav2vec2-base-960h', '22aad52d435eb6dbaf354bdad9b0da84ce7d6156'
TRANSCRIPT = 'I had that curiosity beside me at this moment'

urllib.request.urlretrieve(WAV_URL, 'voices_sp0307.wav')
audio = AudioPart(open('voices_sp0307.wav', 'rb').read(), sample_rate=16000)
aligner = SpectrogramGuidedAligner(Wav2Vec2Ctc(CTC_MODEL, revision=CTC_REVISION), energy_weight=0.8, flux_weight=0.2)
segments = aligner.align(TRANSCRIPT, audio)

pd.DataFrame([{'word': s.token, 'CTC start, s': s.ctc_start_time, 'start, s': s.start_time,
               'end, s': s.end_time, 'CTC confidence': s.confidence, 'boundary refined': s.boundary_refined}
              for s in segments]).round(3)

In the library's figure, dashed lines are the rough CTC boundaries, arrows show where refinement moved them, solid lines are the final word boundaries, and the lower panel is the RMS energy: the cuts land in its dips.

In [ ]:
plot_audio_segments(audio, segments)
plt.show()

Now the mask. A message with audio and a transcript becomes word players through `SgpaSegmentPartitioner`; every player has both a text span and a span of samples. Let us switch off every second word with two policies: `delete` cuts the samples (the recording gets shorter), `neutral` replaces them with zero amplitude and keeps the timing of the other words — this is the silence mask from the lesson. In the transcript text `delete` removes the word, and `neutral` replaces it with underscores of the same length.

In [ ]:
spoken = ConversationSnapshot.from_turns((Turn(messages=(Message(role=Role.USER, content=(TextPart(TRANSCRIPT), audio)),)),))
audio_players = SgpaSegmentPartitioner(aligner=aligner).partition(spoken)
keep = CoalitionMask(tuple(i % 2 == 0 for i in range(audio_players.num_players)))

waves = {}
for policy in (DeletePolicy(), NeutralPolicy()):
    rendered = policy.apply(spoken, audio_players, keep).messages()[0]
    waves[policy.name], rate = decode_audio(rendered.content[1].data())
    print(f'{policy.name:8} {len(waves[policy.name]) / rate:.2f} s | text: {rendered.text!r}')

original, rate = decode_audio(audio.data())
print(f'original recording: {len(original) / rate:.2f} s')
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)
for ax, (title, wave) in zip(axes, [('original recording', original), ('neutral: silence instead of words', waves['neutral']),
                                    ('delete: words cut out', waves['delete'])]):
    ax.plot(np.arange(len(wave)) / rate, wave, linewidth=0.4)
    ax.set_title(title, fontsize=10)
axes[-1].set_xlabel('time, s')
plt.tight_layout()
plt.show()

**Self-check question.** How many transcript words got segments, and for how many of them were both boundaries refined by energy and spectral flux? How long is the original recording, and how long is it after the `delete` policy?

<details><summary>Check yourself</summary>

All 9 transcript words got segments; both boundaries were refined for 7 of them, while the shared boundary of `curiosity` and `beside` was not refined: the search window had no point quiet enough, so the cut stayed in the middle of the CTC gap between the words. The original recording lasts 3.40 s; after `delete` with the words `had`, `curiosity`, `me`, `this` switched off 1.92 s remain, and after `neutral` the length is the same, 3.40 s.

</details>

## An image: PixelSHAP by hand

In the lesson "SHAP on generative models", PixelSHAP (Goldshmidt, 2025) carries TokenSHAP over to images: the players are the objects in the image, "removing" an object means painting over its area, and the value function is the similarity of answers:

$$v(S) = \cos\big(E(y),\ E(y_S)\big),$$

where $y$ is the VLM's answer to the original image, $y_S$ is its answer to the image in which only the objects in $S$ remain, and $E$ is a text embedding. `mllm-shap` 0.3.0 has no images, and the authors' `TokenSHAP` package version 0.2.2 on PyPI does not contain PixelSHAP: in the authors' repository it expects GroundingDINO, SAM and a model behind Ollama or a paid API. So we build the same game ourselves from three small open models that need neither an API key nor a Hugging Face token:

- the segmenter `facebook/mask2former-swin-tiny-coco-panoptic` instead of the GroundingDINO and SAM pair: it directly gives object masks with COCO labels;
- the VLM `HuggingFaceTB/SmolVLM-256M-Instruct` answers a question about the image;
- `sentence-transformers/all-MiniLM-L6-v2` gives answer embeddings for the cosine.

The image is `data/cat_and_dog.jpg` from the course repository: a cat and a dog on a bench by a wall. Next to the similarity game we will also compute a probability game, as in Task 3: $v(S) = \log p(y \mid \text{image}_S)$, the log-probability that the VLM gives to its full answer $y$ on the reduced image.

In [ ]:
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, pipeline
from sentence_transformers import SentenceTransformer

urllib.request.urlretrieve('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg',
                           'cat_and_dog.jpg')
image = Image.open('cat_and_dog.jpg').convert('RGB')
image.thumbnail((768, 768))                    # 768 x 512 instead of 5472 x 3648: masks and the VLM run faster

segmenter = pipeline('image-segmentation', model='facebook/mask2former-swin-tiny-coco-panoptic',
                     revision='df6b1142ff50c3276559d9d78f35f6a579c75a77', device=device)
segments = segmenter(image)
objects = [segment['label'] for segment in segments]
masks = [np.array(segment['mask']) > 0 for segment in segments]
n_objects = len(objects)
for segment, mask in zip(segments, masks):
    print(f'{segment["label"]:<18} score {segment["score"]:.3f}, share of pixels {mask.mean():.3f}')
print(f'pixels outside objects: {1 - np.logical_or.reduce(masks).mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, n_objects + 1, figsize=(3.2 * (n_objects + 1), 2.6))
axes[0].imshow(image)
axes[0].set_title('image')
for ax, name, mask in zip(axes[1:], objects, masks):
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'player: {name}')
for ax in axes:
    ax.axis('off')
plt.show()

The segmenter found four objects, and together they cover the whole image: the dog (`dog`), the cat (`cat`), the bench together with the blanket (COCO label `chair`) and the wall (`wall-other-merged`). These are the players, $2^4 = 16$ coalitions in total, so we will compute the Shapley values exactly. A removed object is painted gray, so for the empty coalition the VLM sees a gray rectangle.

The question to the VLM is the same for all coalitions. Generation is greedy, so the answer to each reduced image is unique.

In [ ]:
VLM_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
VLM_REVISION = '7e3e67edbbed1bf9888184d9df282b700a323964'
QUESTION = 'What animals are in this picture? Answer in one short sentence.'

processor = AutoProcessor.from_pretrained(VLM_ID, revision=VLM_REVISION)
vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, revision=VLM_REVISION, dtype=torch.float32).to(device).eval()
vlm_prompt = processor.apply_chat_template(
    [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': QUESTION}]}], add_generation_prompt=True)
pixels = np.array(image)


def masked_image(present):
    # objects outside the coalition are painted gray, the rest of the image stays as is
    result = pixels.copy()
    for keep, mask in zip(present, masks):
        if not keep:
            result[mask] = 128
    return Image.fromarray(result)


def vlm_inputs(picture):
    return processor(text=vlm_prompt, images=[picture], return_tensors='pt').to(device)


def vlm_answer_ids(picture):
    inputs = vlm_inputs(picture)
    with torch.no_grad():
        output = vlm.generate(**inputs, max_new_tokens=30, do_sample=False)
    return output[0, inputs['input_ids'].shape[1]:]


full_answer_ids = vlm_answer_ids(image)
FULL_ANSWER = processor.decode(full_answer_ids, skip_special_tokens=True).strip()
print('answer to the full image:', repr(FULL_ANSWER))

Now let us go through all 16 coalitions. For each one the VLM answers the reduced image, and we compute both value functions: the cosine between the embeddings of this answer and of the full answer, and the log-probability of the full answer on the reduced image (the answer is substituted whole, one forward pass, as in Task 3). On a Colab CPU this takes about a minute and a half.

In [ ]:
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',
                               revision='1110a243fdf4706b3f48f1d95db1a4f5529b4d41', device=device)
full_embedding = embedder.encode([FULL_ANSWER], normalize_embeddings=True)[0]
special_ids = set(processor.tokenizer.all_special_ids)
answer_target = torch.tensor([[t for t in full_answer_ids.tolist() if t not in special_ids]], device=device)


def full_answer_logprob(picture):
    inputs = vlm_inputs(picture)
    ids = torch.cat([inputs['input_ids'], answer_target], dim=1)
    with torch.no_grad():
        logits = vlm(input_ids=ids, attention_mask=torch.ones_like(ids), pixel_values=inputs['pixel_values'],
                     pixel_attention_mask=inputs['pixel_attention_mask']).logits
    logits = logits[0, inputs['input_ids'].shape[1] - 1:-1].float()  # prediction of each answer token
    return logits.log_softmax(-1).gather(1, answer_target[0][:, None]).sum().item()


pixel_records = {}
for size in range(n_objects + 1):
    for S in combinations(range(n_objects), size):
        picture = masked_image([i in S for i in range(n_objects)])
        text = processor.decode(vlm_answer_ids(picture), skip_special_tokens=True).strip()
        similarity = float(embedder.encode([text], normalize_embeddings=True)[0] @ full_embedding)
        pixel_records[frozenset(S)] = (text, similarity, full_answer_logprob(picture))

pd.DataFrame([{'objects in the image': ', '.join(objects[i] for i in sorted(S)) or '(nothing)', 'VLM answer': text,
               'v: similarity': similarity, 'v: log p(full answer)': logprob}
              for S, (text, similarity, logprob) in pixel_records.items()]).round(4)

We compute the Shapley values of both games with the function `shapley_exact` from Task 1. Next to them we put the first-order approximation that PixelSHAP relies on: the contribution of an object is how much $v$ changes when only this object is removed from the full image, $v(N) - v(N\setminus i)$. It costs $n+1$ VLM calls instead of $2^n$. The lesson quotes the authors' estimate: the first order gives "92% correlation" with the exact Shapley values, without saying on which images and by which measure. Let us check it on our image.

In [ ]:
v_similarity = {S: record[1] for S, record in pixel_records.items()}
v_logprob_image = {S: record[2] for S, record in pixel_records.items()}
everything = frozenset(range(n_objects))


def first_order(v):
    return np.array([v[everything] - v[everything - {i}] for i in range(n_objects)])


pixel_table = pd.DataFrame({
    'phi: similarity': shapley_exact(v_similarity, n_objects),
    'first order: similarity': first_order(v_similarity),
    'phi: log p': shapley_exact(v_logprob_image, n_objects),
    'first order: log p': first_order(v_logprob_image),
}, index=objects)
print(pixel_table.round(4).to_string(), '\n')
for name, v, column in [('similarity', v_similarity, 'phi: similarity'), ('log p', v_logprob_image, 'phi: log p')]:
    print(f'{name}: sum of phi = {pixel_table[column].sum():.4f}, v(N) - v(empty) = {v[everything] - v[frozenset()]:.4f}, '
          f'Pearson of phi and first order = {pearsonr(pixel_table[column], first_order(v))[0]:.3f}')

Let us color the objects of the image by their Shapley values: red raises $v$, blue lowers it, and each game has its own scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, title in [(axes[0], 'phi: similarity', 'PixelSHAP: v(S) = cos(E(y), E(y_S))'),
                          (axes[1], 'phi: log p', 'v(S) = log p(y | image_S)')]:
    heat = np.zeros(pixels.shape[:2])
    for value, mask in zip(pixel_table[column], masks):
        heat[mask] = value
    limit = np.abs(pixel_table[column]).max()
    ax.imshow(image)
    shown = ax.imshow(heat, cmap='RdBu_r', vmin=-limit, vmax=limit, alpha=0.6)
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(shown, ax=ax, fraction=0.03, label='Shapley value of the object')
plt.show()

**Self-check question.** Which object got the largest Shapley value in the similarity game, and what is it? What is the contribution of the bench (`chair`) in this game, and why does the first order not see it? What is the Pearson correlation of the exact values and the first order in each game, and what does this say about the "92% correlation" estimate?

<details><summary>Check yourself</summary>

In the similarity game the largest Shapley value belongs to the cat: $\phi = 0.2637$, the dog has 0.1806 and the wall 0.1041. The bench (`chair`) got $-0.0937$: the VLM describes the image where only the bench is left as "Antelope." and the bench with the dog as "Pug.", and the similarity to the answer "Dogs and cats." drops to 0.2607 and 0.2525. The first order gives the bench 0: when only the bench is removed from the full image, the answer stays "Dogs and cats.", and $v(N)-v(N\setminus\text{chair}) = 0$. The contribution of the bench shows up only in small coalitions, and the first order does not look at them. Efficiency holds in both games: the sum of the values equals $v(N)-v(\varnothing)$ (0.4547 and 9.4595), but in the similarity game what is shared is the similarity to the full answer, not the answer itself.

The Pearson correlation of the exact values and the first order is 0.929 in the similarity game and 0.739 in the $\log p$ game. That the first number is close to the authors' "92%" does not confirm their estimate: it is a correlation over four objects of a single image, on the same 16 reduced images it is 0.739 in the other game, and in the $\log p$ game the first order overstates the contribution of the dog threefold (12.3492 against 3.9132). The numbers were obtained on a CPU; on a Colab GPU the last digits may differ.

</details>

## Conclusion

With `mllm-shap` 0.3.0 we checked both ideas of the lessons. The exact estimator on 7 words needs 128 model runs, and the sum of the Shapley values in every game equals $v(N)-v(\varnothing)$. The complementary estimators approximate the exact answer from a part of the coalitions, but they only pay off when the budget is below $2^n$: the SGPA budget $3n^2$ is smaller than full enumeration only from 8 players on ($3\cdot8^2=192 < 2^8=256$), and with 10 players it is already 300 runs against 1024. Changing the value function on the same model answers changes the ranking of words, because it changes the game itself: similarity to the answer, the model's confidence and the probability of a specific answer are three different questions to the model. For audio, the library cuts speech into words and can replace them with silence, but this version has no model that would answer sound. On an image, PixelSHAP uses the same logic to share the similarity of a VLM answer among objects: the exact values of four objects cost 16 VLM calls, while the first-order approximation with 5 calls does not see the negative contribution of the bench, which shows up only in small coalitions.

**Thank you for your work!**